> **Historical CS data:** This notebook was generated before the IEEE39 FMU added checkpoint support and used single-pass explicit coupling. Its CS curves are superseded by `IEEE39_RollbackNewton_Validation.ipynb`; native and ME results remain useful as historical measurements.

# IEEE39 DPsim inverter comparison: native, FMI ME, and FMI CS

This notebook compares three fresh, closed-loop DPsim runs of the same IEEE39 EMT network. The averaged inverter is at bus 30, the network starts from the case powerflow, and a 100 MW load is switched onto bus 39 at 5 s. Each method runs for 10 s at the same communication/network timestep. Native DPsim is the error baseline for ME and CS at each matching timestep.

The native averaged inverter uses the same P/Q target, current limit, passive current sign, and 5 ms current response as the dual-interface FMU. Its initial current state is set to the FMU's start behavior (the P/Q reference evaluated at the solved powerflow voltage). Errors are computed from full-resolution run logs before those large files are discarded; compact plotting traces are retained. The 10–1000 µs data completed with an earlier two-iteration ME ceiling. The 5 ms ME case failed its residual check at that ceiling and was rerun with residual-based Newton using the default 20-iteration ceiling. The reproduction runner now uses that residual-based setting throughout.

**Regenerate runs** from the repository root with `python3 -u dpsim-fmi/examples/ieee39_inverter/run_ieee39_dpsim_native_me_cs_sweep.py`. The sweep includes 10, 25, 50, 100, 500, 1000, and 5000 µs; 5000 µs is shown as a coarse stress case. The notebook only reads saved metrics and traces.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

root = Path.cwd().resolve()
while root != root.parent and not (root / "dpsim-fmi").is_dir():
    root = root.parent
if not (root / "dpsim-fmi").is_dir():
    raise FileNotFoundError("Open this notebook from a DPsim checkout")
results_dir = (
    root / "build-fmi" / "ieee39-inverter" / "dpsim-native-me-cs-10s-nosubcomponents"
)
metrics_path = results_dir / "metrics.json"
metrics = json.loads(metrics_path.read_text())
print(f"Results: {metrics_path}")
print(metrics["configuration"])

## Full-run errors and timing

In [ ]:
rows = []
for step, case in metrics["cases"].items():
    for method in ("native", "me", "cs"):
        r = case["runtimes"][method]
        comp = case["comparisons"].get(method)
        s = comp["full_run"] if comp else {}
        rows.append(
            {
                "timestep [µs]": int(step),
                "method": method.upper(),
                "PCC voltage RMS error [%]": s.get("voltage_rms_error_percent"),
                "inverter current RMS error [%]": s.get("current_rms_error_percent"),
                "solver wall [s]": r["solver_wall_s"],
                "process wall [s]": r["process_wall_s"],
            }
        )
summary = pd.DataFrame(rows).sort_values(["timestep [µs]", "method"])
display(
    summary.style.format(
        {c: "{:.5g}" for c in summary.select_dtypes("number").columns}, na_rep="—"
    )
)
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for method, group in summary.groupby("method"):
    if method != "NATIVE":
        ax[0].plot(
            group["timestep [µs]"],
            group["PCC voltage RMS error [%]"],
            "o-",
            label=method,
        )
        ax[1].plot(
            group["timestep [µs]"],
            group["inverter current RMS error [%]"],
            "o-",
            label=method,
        )
    ax[2].plot(group["timestep [µs]"], group["solver wall [s]"], "o-", label=method)
for a, title, ylabel in zip(
    ax,
    [
        "PCC voltage error vs native DPsim",
        "Inverter current error vs native DPsim",
        "Solver wall time",
    ],
    [
        "Full-run three-phase RMS error [%]",
        "Full-run three-phase RMS error [%]",
        "Run time [s]",
    ],
):
    a.set_xscale("log")
    a.set_xlabel("Fixed step [µs]")
    a.set_ylabel(ylabel)
    a.set_title(title)
    a.grid(True, alpha=0.3)
    a.legend()
plt.tight_layout()

## Initialization check

In [ ]:
init_rows = []
for step, case in metrics["cases"].items():
    for method, comp in case["comparisons"].items():
        x = comp["initial"]
        init_rows.append(
            {
                "step [µs]": int(step),
                "method": method.upper(),
                "native Vabc at t=0 [V]": x["native_v_abc_V"],
                "candidate Vabc at t=0 [V]": x["candidate_v_abc_V"],
                "native Iabc at t=0 [A]": x["native_i_abc_A"],
                "candidate Iabc at t=0 [A]": x["candidate_i_abc_A"],
            }
        )
display(pd.DataFrame(init_rows))

## Per-timestep cumulative errors and 5 s trajectory detail

For each timestep, the upper panels show cumulative three-phase RMS error from the start of the run through each sample. The lower panels show phase-a PCC voltage and inverter-current trajectories in the saved zoom window around 5 s. These are trajectory details at 5 s; no event marker is drawn. Cumulative traces are estimated from retained compact traces, while the full-run summary above uses full-resolution data.

In [ ]:
import io
from IPython.display import display, Markdown

figure_dir = results_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

for step, case in sorted(metrics["cases"].items(), key=lambda item: int(item[0])):
    step = int(step)
    display(Markdown(f"### {step} µs"))
    me = case["comparisons"]["me"]
    cs = case["comparisons"]["cs"]
    native_path = Path(case["runtimes"]["native"]["directory"]) / "native_trace.csv"
    native = pd.read_csv(native_path)

    fig, ax = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
    for mode, label, color in (
        ("me", "DPsim ME", "tab:blue"),
        ("cs", "DPsim CS", "tab:orange"),
    ):
        candidate = pd.read_csv(case["comparisons"][mode]["compact_trace"])
        if len(candidate) != len(native) or not np.allclose(
            candidate.time_s, native.time_s, rtol=0, atol=1e-12
        ):
            raise ValueError(
                f"{step} µs {label} compact trace does not align with native DPsim"
            )
        vcols = ["pcc_v_a_V", "pcc_v_b_V", "pcc_v_c_V"]
        icols = ["current_a_A", "current_b_A", "current_c_A"]
        vdiff = (candidate[vcols].to_numpy() - native[vcols].to_numpy()) ** 2
        idiff = (candidate[icols].to_numpy() - native[icols].to_numpy()) ** 2
        vref = native[vcols].to_numpy() ** 2
        iref = native[icols].to_numpy() ** 2
        verror = 100 * np.sqrt(
            np.cumsum(vdiff.sum(axis=1)) / np.cumsum(vref.sum(axis=1))
        )
        ierror = 100 * np.sqrt(
            np.cumsum(idiff.sum(axis=1)) / np.cumsum(iref.sum(axis=1))
        )
        ax[0, 0].plot(native.time_s, verror, color=color, label=label)
        ax[0, 1].plot(native.time_s, ierror, color=color, label=label)

    zoom_me = pd.read_csv(me["load_step_zoom_trace"])
    zoom_cs = pd.read_csv(cs["load_step_zoom_trace"])
    ax[1, 0].plot(zoom_me.time_s, zoom_me.native_v_a_V, "k--", label="Native DPsim")
    ax[1, 0].plot(zoom_me.time_s, zoom_me.me_v_a_V, color="tab:blue", label="DPsim ME")
    ax[1, 0].plot(
        zoom_cs.time_s, zoom_cs.cs_v_a_V, color="tab:orange", label="DPsim CS"
    )
    ax[1, 1].plot(zoom_me.time_s, zoom_me.native_i_a_A, "k--", label="Native DPsim")
    ax[1, 1].plot(zoom_me.time_s, zoom_me.me_i_a_A, color="tab:blue", label="DPsim ME")
    ax[1, 1].plot(
        zoom_cs.time_s, zoom_cs.cs_i_a_A, color="tab:orange", label="DPsim CS"
    )

    ax[0, 0].set_title("Cumulative PCC voltage RMS error")
    ax[0, 1].set_title("Cumulative inverter current RMS error")
    ax[1, 0].set_title("PCC phase-a voltage near 5 s")
    ax[1, 1].set_title("Inverter phase-a current near 5 s")
    ax[0, 0].set_ylabel("Cumulative RMS error [%]")
    ax[0, 1].set_ylabel("Cumulative RMS error [%]")
    ax[1, 0].set_ylabel("Voltage [V]")
    ax[1, 1].set_ylabel("Current [A]")
    for axis in ax.flat:
        axis.set_xlabel("Time [s]")
        axis.grid(True, alpha=0.3)
        axis.legend()
    fig.suptitle(f"IEEE39, {step} µs: cumulative error and trajectory detail")
    image_buffer = io.BytesIO()
    fig.savefig(image_buffer, format="png", dpi=150)
    image_bytes = image_buffer.getvalue()
    (figure_dir / f"cumulative_and_5s_trajectory_{step}us.png").write_bytes(image_bytes)
    display(fig)
    plt.close(fig)

## Interpretation notes

Each timestep is a separate comparison and uses the native DPsim run at that same timestep as its baseline. The percentages describe implementation agreement at a matched discretization, not absolute physical accuracy. A coarse timestep changes the discrete-time network and inverter trajectories for all three methods. The 5000 µs result is a coarse-step stress test; interpret it separately from the 10–1000 µs sweep.